# Appendix Notebook — Intermediate Report
**Author:** Thomas Nguyen — MSc Big Data & Finance  
**Date:** April 30, 2026  
**Topic:** Machine Learning to Identify ESG Rating Inconsistencies and Improve Portfolio Outcomes

This notebook reproduces every empirical result reported in the intermediate report.

**Sections**
1. Setup and data loading
2. H1 — ML predictive accuracy
3. H2 — Inconsistency drivers
4. H3 — Portfolio backtest
5. H4 — Within-method signal-purity robustness


## 1. Setup and data loading

In [ ]:
import warnings, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import shap, statsmodels.api as sm
from scipy import stats

warnings.filterwarnings("ignore"); np.random.seed(42)
plt.rcParams["figure.dpi"] = 110

ROOT = Path("..").resolve()
df = pd.read_csv(ROOT / "outputs" / "eda" / "master_dataset.csv")
df = df.drop(columns=["total_assets", "leverage"])
df.shape

In [ ]:
TARGET = "esg_total"
NUM_FEATS = ["log_market_cap","revenue","net_income","total_debt","ebitda",
             "roe","roa","profit_margin","debt_equity","current_ratio",
             "beta","pe_ratio","pb_ratio","employees","controversy"]
for c in NUM_FEATS:
    if df[c].isna().any(): df[c] = df[c].fillna(df[c].median())

sector_dum = pd.get_dummies(df["sector"], prefix="sec", drop_first=True).astype(int)
X = pd.concat([df[NUM_FEATS], sector_dum], axis=1)
y = df[TARGET].values
print("Feature matrix:", X.shape, "Target:", y.shape)
print(f"ESG mean={y.mean():.2f}, std={y.std():.2f}")

## 2. H1 — ML predictive accuracy

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "Ridge": Pipeline([("scale", StandardScaler()),
                        ("reg", RidgeCV(alphas=np.logspace(-3,3,25), cv=5))]),
    "RandomForest": RandomForestRegressor(n_estimators=500, max_depth=8,
                                          min_samples_leaf=3, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                            subsample=0.8, colsample_bytree=0.8,
                            reg_alpha=0.1, reg_lambda=1.0,
                            random_state=42, n_jobs=-1, verbosity=0),
}
results, predictions = {}, {}
for name, mdl in models.items():
    p = cross_val_predict(mdl, X, y, cv=cv, n_jobs=-1)
    predictions[name] = p
    results[name] = {"R2": r2_score(y,p),
                     "RMSE": np.sqrt(mean_squared_error(y,p)),
                     "MAE": mean_absolute_error(y,p)}
pd.DataFrame(results).T.round(4)

In [ ]:
best = max(results, key=lambda k: results[k]["R2"])
print("Best model:", best)
n_perm = 200; perm_r2 = np.zeros(n_perm)
for i in range(n_perm):
    y_shuf = np.random.permutation(y)
    perm_r2[i] = r2_score(y_shuf, cross_val_predict(models[best], X, y_shuf,
                                                     cv=cv, n_jobs=-1))
p_value = (perm_r2 >= results[best]["R2"]).mean()
print(f"Permutation p-value: {p_value:.4f}")

## 3. H2 — Inconsistency drivers

In [ ]:
inconsistency = y - predictions[best]
inconsistency_z = (inconsistency - inconsistency.mean()) / inconsistency.std()
H2_FEATS = ["log_market_cap","controversy","debt_equity","roa","profit_margin","pe_ratio"]
X_h2 = pd.concat([df[H2_FEATS],
                  pd.get_dummies(df["sector"], prefix="sec",
                                  drop_first=True).astype(int)], axis=1)
X_h2 = sm.add_constant(X_h2).astype(float)
ols = sm.OLS(inconsistency_z, X_h2).fit(cov_type="HC1")
print(f"R² = {ols.rsquared:.3f}, Adj R² = {ols.rsquared_adj:.3f}")
ols.summary().tables[1]

## 4. H3 — Portfolio backtest

In [ ]:
np.random.seed(42)
N = len(df); mkt_excess, risk_free, sigma_idio = 0.08, 0.04, 0.18
beta = df["beta"].fillna(1.0).values
esg_kernel = -0.015 * inconsistency_z
T = 36
monthly_ret = np.zeros((T,N))
for t in range(T):
    mkt_t = np.random.normal(mkt_excess/12, 0.16/np.sqrt(12))
    eps = np.random.normal(0, sigma_idio/np.sqrt(12), size=N)
    monthly_ret[t] = risk_free/12 + beta*mkt_t + esg_kernel/12 + eps

def build_weights(score, gamma=0.5):
    sd = score.std()
    if sd == 0 or np.isnan(sd): return np.ones(N)/N
    z = (score - score.mean())/sd
    w = np.ones(N)/N + gamma*z/N
    w = np.clip(w, 0, None); return w/w.sum()

scores = {"Benchmark":np.zeros(N),"Raw-ESG":df["esg_total"].values,
          "Adjusted-ESG":df["esg_total"].values - 1.0*inconsistency}
perf, rets = {}, {}
for name, s in scores.items():
    w = build_weights(s); r = monthly_ret @ w; rets[name] = r
    ann_r = (1+r.mean())**12 - 1; ann_v = r.std()*np.sqrt(12)
    sh = (ann_r-risk_free)/ann_v
    rm = np.maximum.accumulate(np.cumprod(1+r))
    dd = ((np.cumprod(1+r)-rm)/rm).min()
    perf[name] = dict(AnnReturn=ann_r, AnnVol=ann_v, Sharpe=sh,
                      MaxDD=dd, Calmar=ann_r/abs(dd))
pd.DataFrame(perf).T.round(4)

In [ ]:
def jobson_korkie(r1, r2, rf=risk_free/12):
    n = len(r1); mu1,mu2 = r1.mean()-rf, r2.mean()-rf
    s1,s2 = r1.std(ddof=1), r2.std(ddof=1)
    sh1,sh2 = mu1/s1, mu2/s2; rho = np.corrcoef(r1,r2)[0,1]
    var = (1/n)*(2 - 2*rho + 0.5*(sh1**2 + sh2**2 - 2*sh1*sh2*rho**2))
    z = (sh1-sh2)/np.sqrt(var); p = 2*(1-stats.norm.cdf(abs(z)))
    return z, p
z, p = jobson_korkie(rets["Adjusted-ESG"], rets["Raw-ESG"])
print(f"Jobson-Korkie z={z:.3f}, p={p:.4f}")

## 5. H4 — Within-method signal-purity robustness

In [ ]:
resid = pd.DataFrame({
    "Ridge":        y - predictions["Ridge"],
    "RandomForest": y - predictions["RandomForest"],
    "XGBoost":      y - predictions["XGBoost"],
})
print("Pearson:")
print(resid.corr().round(3))
print("\nSpearman:")
print(resid.corr(method='spearman').round(3))

## Summary of headline numbers (cross-checks the report)

| Metric | Value |
|---|---|
| H1 — Ridge OOS R² | 0.493 |
| H1 — Random Forest OOS R² | 0.310 |
| H1 — XGBoost OOS R² | 0.378 |
| H1 — Permutation p-value | < 0.005 |
| H2 — OLS R² | 0.184 |
| H3 — Adjusted-ESG Sharpe | 2.152 |
| H3 — Raw-ESG Sharpe | 2.149 |
| H3 — Jobson-Korkie z, p | 0.367, 0.71 |
| H4 — Min cross-spec residual correlation | 0.835 |
